# One Bottleneck, Four Superpowers
### An Interactive Tour of Auto-encoders

An **auto-encoder** learns *without labels*: it simply tries to copy its input through a narrow **bottleneck**.

    input  ->  [ Encoder ]  ->  code z  ->  [ Decoder ]  ->  reconstruction
    28x28       compress      low-dim        expand            28x28

Because the code is far smaller than the input, the network **cannot just memorize an identity map** -- it is forced to discover the underlying *structure* of the data. That single idea powers four seemingly different capabilities, and that is the story of this notebook:

1. **Reconstruction** -- train one small auto-encoder on Fashion-MNIST and watch the bottleneck rebuild images.
2. **Latent codes as dimension reduction** -- read the bottleneck as a learned embedding and compare it with **PCA** and **t-SNE**.
3. **De-noising & the BERT connection** -- corrupt the input, reconstruct the clean original, and see why masked language modeling *is* a de-noising auto-encoder.
4. **Anomaly detection** -- use reconstruction error as an anomaly score for one-class data.

> **One backbone, trained once.** A single small auto-encoder (trained in Concept 1) is reused for Concepts 2 and 4; Concept 3 needs only a *one-line* corruption change. The whole notebook runs end-to-end in a couple of minutes on a free Colab CPU/GPU.

Every concept below is a **variant of one mechanism: reconstruction through a bottleneck.**

In [ ]:
# Environment & reproducibility setup
import os, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import roc_curve, roc_auc_score

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    WIDGETS_OK = True
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])
    try:
        import ipywidgets as widgets
        from ipywidgets import interact
        WIDGETS_OK = True
    except Exception as e:
        WIDGETS_OK = False
        print('ipywidgets unavailable; interactive cells fall back to a static view.', e)

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

plt.rcParams['figure.dpi'] = 110
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['figure.figsize'] = (8, 3)
plt.rcParams['axes.grid'] = False

# Global hyperparameters reused everywhere downstream
SEED = 42
BATCH_SIZE = 256
EPOCHS = 8
LATENT_DIM = 32
LR = 1e-3
NOISE_SIGMA = 0.5

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| device:', device)

# sanity checks
assert isinstance(torch.__version__, str) and len(torch.__version__) > 0
set_seed(SEED); _a = torch.randn(3)
set_seed(SEED); _b = torch.randn(3)
assert torch.allclose(_a, _b)
assert LATENT_DIM > 0 and EPOCHS > 0 and 0.0 <= NOISE_SIGMA <= 1.0
print('Setup OK.')

In [ ]:
# Fashion-MNIST + fixed one-class (normal vs anomalous) split
transform = transforms.ToTensor()  # -> float (1,28,28) already scaled to [0,1]

try:
    train_set = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
    test_set  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
except Exception as e:
    print('Fashion-MNIST download failed (network/SSL/quota). Please re-run this cell.')
    raise

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

normal_classes    = [0, 1, 2, 3, 4, 5, 6, 7]
anomalous_classes = [8, 9]                       # Bag, Ankle boot: never seen in training
assert set(normal_classes).isdisjoint(anomalous_classes)
assert all(0 <= c <= 9 for c in normal_classes + anomalous_classes)

N_NORMAL_TEST = 2000
N_ANOM_TEST   = 1000

train_x = train_set.data.float().unsqueeze(1) / 255.0
train_y = train_set.targets
test_x  = test_set.data.float().unsqueeze(1) / 255.0
test_y  = test_set.targets

def make_subset_tensor(images, labels, keep_classes, n=None, seed=SEED):
    mask = torch.zeros(len(labels), dtype=torch.bool)
    for c in keep_classes:
        mask |= (labels == c)
    assert mask.sum() > 0, 'no samples for requested classes'
    imgs, labs = images[mask], labels[mask]
    if n is not None:
        n = min(n, imgs.size(0))                 # clamp so we never index out of range
        set_seed(seed)
        perm = torch.randperm(imgs.size(0))[:n]
        imgs, labs = imgs[perm], labs[perm]
    return imgs, labs

normal_train_mask = torch.zeros(len(train_y), dtype=torch.bool)
for c in normal_classes:
    normal_train_mask |= (train_y == c)
assert normal_train_mask.sum() > 0
train_loader = DataLoader(TensorDataset(train_x[normal_train_mask]),
                          batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

normal_test_images, normal_test_labels = make_subset_tensor(test_x, test_y, normal_classes,    N_NORMAL_TEST)
anomalous_test_images, _               = make_subset_tensor(test_x, test_y, anomalous_classes, N_ANOM_TEST)

set_seed(SEED)
sample_grid_figure, axes = plt.subplots(2, 5, figsize=(9, 4))
for c, ax in zip(range(10), axes.ravel()):
    idx = (test_y == c).nonzero().flatten()[0].item()
    ax.imshow(test_x[idx].squeeze(), cmap='gray')
    tag = 'normal' if c in normal_classes else 'ANOMALY'
    ax.set_title(class_names[c] + ' (' + tag + ')', fontsize=8)
    ax.axis('off')
sample_grid_figure.suptitle('Fashion-MNIST: normal (train) vs held-out anomalous classes', y=1.02)
plt.tight_layout(); plt.show()

# sanity checks
assert normal_test_images.min() >= 0.0 and normal_test_images.max() <= 1.0
assert normal_test_images.shape[1:] == (1, 28, 28) and anomalous_test_images.shape[1:] == (1, 28, 28)
assert normal_test_images.shape[0] == normal_test_labels.shape[0]
assert len(train_loader.dataset) > 0
assert all(int(l) in normal_classes for l in normal_test_labels)
print('train(normal-only):', len(train_loader.dataset),
      '| normal-test:', len(normal_test_images),
      '| anomalous-test:', len(anomalous_test_images))

## Concept 1 -- The Basic Auto-encoder & Reconstruction

An auto-encoder is two networks trained together:

- **Encoder** f: compresses the input x (a 28x28 image) into a small **code** z = f(x) -- the *bottleneck*.
- **Decoder** g: reconstructs the image x_hat = g(z) from that code.

We train it so the output is *as close as possible* to the input by minimizing the **reconstruction (MSE) loss**:

> L  =  (1/N) Σ_i  || x_i  -  g(f(x_i)) ||²

**Why the bottleneck matters.** If the code were as large as the input, the network could trivially copy the input (the identity function) and learn nothing. Forcing z to be *low-dimensional* makes the auto-encoder discover the **structure** that explains the data -- the recurring shapes, textures, and parts of clothing items.

The resulting code is a **learned representation / embedding** (Hinton & Salakhutdinov, 2006) that we reuse for every downstream task in this notebook.

In [ ]:
# The single reusable backbone: AutoEncoder + a generic training loop
class AutoEncoder(nn.Module):
    def __init__(self, latent_dim: int = LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64),  nn.ReLU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 256),        nn.ReLU(),
            nn.Linear(256, 784),       nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)                       # [B, latent_dim]

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)   # [B,1,28,28] in [0,1]

    def forward(self, x):
        return self.decode(self.encode(x))

def train_ae(model, loader, corrupt_fn=None, epochs=EPOCHS, lr=LR, device=device, verbose=True):
    assert len(loader) > 0, 'empty loader: no training would happen'
    model.to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = []
    for epoch in range(1, epochs + 1):
        running, nb = 0.0, 0
        for batch in loader:
            (x,) = batch                  # TensorDataset yields a 1-tuple
            x = x.to(device)
            target = x                    # clean target, even when the input is corrupted
            inp = corrupt_fn(x) if corrupt_fn is not None else x
            out = model(inp)
            loss = criterion(out, target)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item(); nb += 1
        history.append(running / max(nb, 1))
        if verbose:
            print('  epoch', epoch, '/', epochs, ' loss=%.5f' % history[-1])
    return history

# sanity checks
_m = AutoEncoder(); _x = torch.rand(4, 1, 28, 28)
assert _m(_x).shape == _x.shape and _m.encode(_x).shape == (4, LATENT_DIM)
assert _m(_x).min() >= 0.0 and _m(_x).max() <= 1.0
_opt = torch.optim.Adam(_m.parameters(), lr=LR)
for _ in range(2):
    _loss = nn.MSELoss()(_m(_x), _x); _opt.zero_grad(); _loss.backward(); _opt.step()
assert math.isfinite(_loss.item())
print('AutoEncoder + train_ae ready.')

In [ ]:
# Train the ONE core auto-encoder (reused by concepts 1, 2, 4)
set_seed(SEED)
core_ae = AutoEncoder(latent_dim=LATENT_DIM).to(device)
try:
    core_loss_history = train_ae(core_ae, train_loader, corrupt_fn=None, epochs=EPOCHS)
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('CUDA OOM: falling back to CPU and retrying once.')
        torch.cuda.empty_cache()
        device = torch.device('cpu')
        set_seed(SEED)
        core_ae = AutoEncoder(latent_dim=LATENT_DIM).to(device)
        core_loss_history = train_ae(core_ae, train_loader, corrupt_fn=None, epochs=EPOCHS, device=device)
    else:
        raise

core_ae.eval()
assert len(core_loss_history) > 0
loss_curve_figure = plt.figure(figsize=(5, 3))
plt.plot(range(1, len(core_loss_history) + 1), core_loss_history, marker='o')
plt.xlabel('epoch'); plt.ylabel('MSE reconstruction loss')
plt.title('Core auto-encoder training'); plt.tight_layout(); plt.show()
print('Final core reconstruction loss: %.5f' % core_loss_history[-1])

# sanity checks
assert len(core_loss_history) == EPOCHS and all(math.isfinite(v) for v in core_loss_history)
assert core_loss_history[-1] < core_loss_history[0]
assert not core_ae.training

In [ ]:
# Shared inference helper + static reconstructions
def reconstruct(model, images):
    model.eval()
    if images.dim() == 3:
        images = images.unsqueeze(0)
    with torch.no_grad():
        return model(images.to(device)).detach().cpu()

batch = normal_test_images[:8]
recons = reconstruct(core_ae, batch)
reconstruction_figure, axes = plt.subplots(2, 8, figsize=(11, 3))
for j in range(8):
    axes[0, j].imshow(batch[j].squeeze(), cmap='gray'); axes[0, j].axis('off')
    axes[1, j].imshow(recons[j].squeeze(), cmap='gray'); axes[1, j].axis('off')
axes[0, 0].set_title('input', loc='left', fontsize=9)
axes[1, 0].set_title('reconstruction', loc='left', fontsize=9)
reconstruction_figure.suptitle('Core AE: top row input, bottom row reconstruction')
plt.tight_layout(); plt.show()

baseline_recon_error = ((reconstruct(core_ae, normal_test_images) - normal_test_images) ** 2).mean().item()
print('Baseline mean reconstruction MSE (normal-test): %.5f' % baseline_recon_error)

# sanity checks
assert reconstruct(core_ae, normal_test_images[:8]).shape == normal_test_images[:8].shape
assert 0.0 <= baseline_recon_error and math.isfinite(baseline_recon_error)
_r = reconstruct(core_ae, normal_test_images[:8])
assert _r.min() >= 0 and _r.max() <= 1

In [ ]:
# Interactive reconstruction explorer
class_to_idx = {c: (normal_test_labels == c).nonzero().flatten().tolist() for c in normal_classes}
_name_to_class = {class_names[c]: c for c in normal_classes}

def show_reconstruction(index=0, class_choice='(all)'):
    if class_choice != '(all)':
        idx_list = class_to_idx[_name_to_class[class_choice]]
        idx = idx_list[index % len(idx_list)]
    else:
        idx = index % len(normal_test_images)
    img = normal_test_images[idx:idx + 1]
    rec = reconstruct(core_ae, img)
    err = ((rec - img) ** 2).mean().item()
    label = class_names[int(normal_test_labels[idx])]
    fig, ax = plt.subplots(1, 2, figsize=(5, 2.6))
    ax[0].imshow(img.squeeze(), cmap='gray'); ax[0].set_title('input'); ax[0].axis('off')
    ax[1].imshow(rec.squeeze(), cmap='gray'); ax[1].set_title('reconstruction'); ax[1].axis('off')
    fig.suptitle('class=%s  index=%d  MSE=%.4f' % (label, idx, err))
    plt.tight_layout(); plt.show()

show_reconstruction(0, '(all)')   # default static render (useful even without widgets)

reconstruction_explorer_widget = None
if WIDGETS_OK:
    try:
        reconstruction_explorer_widget = interact(
            show_reconstruction,
            index=widgets.IntSlider(min=0, max=len(normal_test_images) - 1, step=1, value=0),
            class_choice=widgets.Dropdown(options=['(all)'] + [class_names[c] for c in normal_classes], value='(all)'),
        )
    except Exception as e:
        print('Widget init failed; static view shown above.', e)

# sanity checks
assert all(len(v) > 0 for v in class_to_idx.values())
show_reconstruction(len(normal_test_images) - 1, class_names[normal_classes[0]])

## Concept 2 -- The Latent Code as Dimension Reduction

The bottleneck code z is not just an intermediate value -- it is a **new, low-dimensional feature** we can hand to downstream tasks. The encoder *finds patterns and then compresses*.

How does this learned, **non-linear** code compare with the classic, *non-deep* dimension-reduction tools?

- **PCA** -- a *linear* projection onto the directions of greatest variance. Fast, but it can only rotate and scale the pixel space.
- **t-SNE** -- a *non-linear* method that preserves local neighborhoods; excellent for visualization, but it is a visualization tool, not a reusable encoder.

We encode the test set with our trained auto-encoder and show three 2-D maps side by side -- **AE code (via t-SNE)**, **PCA of the AE code**, and **PCA of the raw pixels** -- to see whether the auto-encoder's code separates clothing classes more cleanly than a purely linear pixel projection. Then we make the code *tangible* by **interpolating** inside the latent space.

In [ ]:
# Representation comparison: AE code vs PCA-of-code vs PCA-of-pixels
set_seed(SEED)
N_EMB = min(1000, len(normal_test_images))
imgs = normal_test_images[:N_EMB]
labels = normal_test_labels[:N_EMB].numpy()
with torch.no_grad():
    test_codes = core_ae.encode(imgs.to(device)).cpu().numpy()      # [N_EMB, LATENT_DIM]

if LATENT_DIM == 2:
    ae_embedding_2d = test_codes
    titleA = 'AE code (2-D)'
else:
    perplexity = min(30, max(5, N_EMB // 100))
    assert perplexity < N_EMB
    try:
        ae_embedding_2d = TSNE(n_components=2, perplexity=perplexity,
                               init='pca', random_state=SEED).fit_transform(test_codes)
        titleA = 'AE code (t-SNE)'
    except Exception as e:
        print('t-SNE failed; using PCA for panel A.', e)
        ae_embedding_2d = PCA(n_components=2, random_state=SEED).fit_transform(test_codes)
        titleA = 'AE code (PCA fallback)'

pca_codes_2d = PCA(n_components=2, random_state=SEED).fit_transform(test_codes)
pixel_pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(imgs.reshape(N_EMB, -1).numpy())

embedding_comparison_figure, axes = plt.subplots(1, 3, figsize=(13, 4))
panels = [(ae_embedding_2d, titleA), (pca_codes_2d, 'AE code (PCA)'), (pixel_pca_2d, 'Raw pixels (PCA)')]
for ax, (emb, title) in zip(axes, panels):
    ax.scatter(emb[:, 0], emb[:, 1], c=labels, cmap='tab10', s=8, vmin=0, vmax=9)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
handles = [plt.Line2D([0], [0], marker='o', linestyle='', markersize=6, color=plt.cm.tab10(c / 9.0))
           for c in normal_classes]
embedding_comparison_figure.legend(handles, [class_names[c] for c in normal_classes],
                                   loc='center right', fontsize=7, title='class')
embedding_comparison_figure.suptitle('AE codes separate classes more cleanly than linear pixel PCA')
plt.tight_layout(rect=[0, 0, 0.9, 1]); plt.show()

# sanity checks
assert test_codes.shape == (N_EMB, LATENT_DIM) and ae_embedding_2d.shape == (N_EMB, 2)
assert np.isfinite(ae_embedding_2d).all() and np.isfinite(pixel_pca_2d).all()
assert len(np.unique(labels)) >= 2

In [ ]:
# Interactive latent-space interpolation (morph)
compression_ratio = 784 // LATENT_DIM
print('Compression: 784 pixels ->', LATENT_DIM, 'latent dims (about', compression_ratio, 'x smaller)')
N = len(normal_test_images)

def interpolate(i=0, j=1, alpha=0.5):
    i = int(i) % N
    j = int(j) % N
    if j == i:
        j = (i + 1) % N
    with torch.no_grad():
        z1 = core_ae.encode(normal_test_images[i:i + 1].to(device))
        z2 = core_ae.encode(normal_test_images[j:j + 1].to(device))
        morph = core_ae.decode((1 - alpha) * z1 + alpha * z2).cpu()
    fig, ax = plt.subplots(1, 3, figsize=(6, 2.6))
    ax[0].imshow(normal_test_images[i].squeeze(), cmap='gray'); ax[0].set_title('source i=%d' % i); ax[0].axis('off')
    ax[1].imshow(morph.squeeze(), cmap='gray'); ax[1].set_title('morph alpha=%.2f' % alpha); ax[1].axis('off')
    ax[2].imshow(normal_test_images[j].squeeze(), cmap='gray'); ax[2].set_title('target j=%d' % j); ax[2].axis('off')
    plt.tight_layout(); plt.show()

interpolate(0, 1, 0.5)   # default static render

latent_interpolation_widget = None
if WIDGETS_OK:
    try:
        latent_interpolation_widget = interact(
            interpolate,
            i=widgets.IntSlider(min=0, max=N - 1, value=0),
            j=widgets.IntSlider(min=0, max=N - 1, value=1),
            alpha=widgets.FloatSlider(min=0.0, max=1.0, step=0.1, value=0.5),
        )
    except Exception as e:
        print('Widget init failed; static morph shown above.', e)

# sanity checks
_d = ((core_ae.decode(core_ae.encode(normal_test_images[0:1].to(device))).cpu()
       - reconstruct(core_ae, normal_test_images[0:1])) ** 2).mean().item()
assert _d < 1e-5
assert compression_ratio > 1

## Concept 3 -- The De-noising Auto-encoder (and its link to BERT)

A **de-noising auto-encoder** (Vincent et al., 2008) changes *one thing*: we **corrupt the input** (add noise or mask pixels) but still require the decoder to reproduce the **original clean image**.

    clean x  ->  corrupt  ->  noisy x~  ->  [ Encoder -> code -> Decoder ]  ->  x_hat     (target = clean x)

Why does this help? Copying is no longer an option -- to undo the corruption the network must learn what a *plausible, clean* clothing item looks like, so it is forced to capture **meaningful structure**.

**The BERT connection.** This is exactly how modern self-supervised language models pre-train. **BERT is a de-noising auto-encoder**: the noise is *masking* some tokens, and the model is trained to reconstruct the original tokens. Same recipe -- corrupt, then reconstruct the clean original -- at the scale of text.

In code this is a **one-line change**: we reuse the *same* AutoEncoder class and train_ae loop, only passing a corrupt_fn.

In [ ]:
# Train a de-noising AE by REUSING the C05 class + loop
def add_noise(x, sigma=NOISE_SIGMA):
    return torch.clamp(x + sigma * torch.randn_like(x), 0.0, 1.0)

set_seed(SEED)
denoise_ae = AutoEncoder(latent_dim=LATENT_DIM).to(device)
try:
    denoise_loss_history = train_ae(denoise_ae, train_loader,
                                    corrupt_fn=lambda b: add_noise(b, NOISE_SIGMA), epochs=EPOCHS)
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('CUDA OOM: falling back to CPU and retrying once.')
        torch.cuda.empty_cache()
        device = torch.device('cpu')
        set_seed(SEED)
        denoise_ae = AutoEncoder(latent_dim=LATENT_DIM).to(device)
        denoise_loss_history = train_ae(denoise_ae, train_loader,
                                        corrupt_fn=lambda b: add_noise(b, NOISE_SIGMA), epochs=EPOCHS, device=device)
    else:
        raise

denoise_ae.eval()
plt.figure(figsize=(5, 3))
plt.plot(range(1, len(denoise_loss_history) + 1), denoise_loss_history, marker='o', color='tab:orange')
plt.xlabel('epoch'); plt.ylabel('MSE (corrupted -> clean)')
plt.title('De-noising auto-encoder training'); plt.tight_layout(); plt.show()
print('Final de-noising loss: %.5f' % denoise_loss_history[-1])

# sanity checks
_n = add_noise(normal_test_images[:4], 0.5)
assert _n.shape == normal_test_images[:4].shape and _n.min() >= 0 and _n.max() <= 1
assert not torch.allclose(_n, normal_test_images[:4])
assert len(denoise_loss_history) == EPOCHS and denoise_loss_history[-1] < denoise_loss_history[0]

In [ ]:
# Static clean -> noisy -> denoised demo (+ reusable wrapper)
def denoise_fn(images, sigma=NOISE_SIGMA):
    if images.dim() == 3:
        images = images.unsqueeze(0)
    noisy = add_noise(images, sigma)
    cleaned = reconstruct(denoise_ae, noisy)
    return noisy.cpu(), cleaned

set_seed(SEED)
clean = normal_test_images[:8]
noisy, denoised = denoise_fn(clean, NOISE_SIGMA)
denoise_demo_figure, axes = plt.subplots(3, 8, figsize=(11, 4))
row_imgs = [clean, noisy, denoised]
row_names = ['clean', 'noisy', 'denoised']
for r in range(3):
    for j in range(8):
        axes[r, j].imshow(row_imgs[r][j].squeeze(), cmap='gray'); axes[r, j].axis('off')
    axes[r, 0].set_title(row_names[r], loc='left', fontsize=9)
denoise_demo_figure.suptitle('De-noising rows: clean / noisy / denoised (sigma=%.2f)' % NOISE_SIGMA)
plt.tight_layout(); plt.show()

# sanity checks
nz, dn = denoise_fn(normal_test_images[:8])
assert nz.shape == dn.shape == normal_test_images[:8].shape
assert ((dn - normal_test_images[:8]) ** 2).mean() < ((nz - normal_test_images[:8]) ** 2).mean()
assert dn.min() >= 0 and dn.max() <= 1

In [ ]:
# Interactive noise-level explorer for de-noising
def show_denoise(index=0, sigma=NOISE_SIGMA):
    index = int(index) % len(normal_test_images)
    sigma = float(min(max(sigma, 0.0), 1.0))
    set_seed(SEED + index)                      # stable corruption per index (no flicker)
    clean = normal_test_images[index:index + 1]
    noisy, denoised = denoise_fn(clean, sigma)
    err = ((denoised - clean) ** 2).mean().item()
    fig, ax = plt.subplots(1, 3, figsize=(6, 2.6))
    ax[0].imshow(clean.squeeze(), cmap='gray'); ax[0].set_title('clean'); ax[0].axis('off')
    ax[1].imshow(noisy.squeeze(), cmap='gray'); ax[1].set_title('noisy sigma=%.2f' % sigma); ax[1].axis('off')
    ax[2].imshow(denoised.squeeze(), cmap='gray'); ax[2].set_title('denoised'); ax[2].axis('off')
    fig.suptitle('index=%d  recon MSE=%.4f' % (index, err))
    plt.tight_layout(); plt.show()

show_denoise(0, NOISE_SIGMA)   # default static render

denoising_explorer_widget = None
if WIDGETS_OK:
    try:
        denoising_explorer_widget = interact(
            show_denoise,
            index=widgets.IntSlider(min=0, max=len(normal_test_images) - 1, value=0),
            sigma=widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=NOISE_SIGMA),
        )
    except Exception as e:
        print('Widget init failed; static view shown above.', e)

# sanity checks
def _err_at(idx, s):
    set_seed(SEED + idx)
    c = normal_test_images[idx:idx + 1]
    _, d = denoise_fn(c, s)
    return ((d - c) ** 2).mean().item()
assert _err_at(0, 0.0) <= _err_at(0, 0.8) + 1e-3
show_denoise(len(normal_test_images) - 1, 1.0)

## Concept 4 -- Anomaly Detection via Reconstruction Error

Here is a trick that needs **no anomaly labels at all**. We already trained core_ae on **normal data only**, so:

- A **normal** input is reconstructed well  ->  **low** reconstruction error.
- An **anomalous** input (something the auto-encoder has never seen) is reconstructed poorly  ->  **high** reconstruction error.

That makes **per-sample reconstruction error a ready-made anomaly score** -- and it reuses the existing core_ae at essentially no extra cost.

This is the **one-class** setting that matters in the real world, where abnormal examples are rare or expensive to label: **fraud detection, network-intrusion detection, and medical (e.g. cancer) screening**. Below we score the held-out anomalous classes (Bag, Ankle boot) against normal items, inspect the error **histograms**, quantify separability with an **ROC curve / AUC**, and finish with an interactive **threshold** that exposes the precision-recall trade-off.

In [ ]:
# Anomaly scoring: per-sample reconstruction error
def recon_error(model, images, batch=512):
    model.eval()
    errs = []
    with torch.no_grad():
        for k in range(0, len(images), batch):
            x = images[k:k + batch].to(device)
            e = ((model(x) - x) ** 2).reshape(x.size(0), -1).mean(dim=1)
            errs.append(e.cpu())
    return torch.cat(errs)

normal_errors  = recon_error(core_ae, normal_test_images)
anomaly_errors = recon_error(core_ae, anomalous_test_images)
assert len(normal_errors) > 0 and len(anomaly_errors) > 0

y_true = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(anomaly_errors))])
scores = np.concatenate([normal_errors.numpy(), anomaly_errors.numpy()])
fpr, tpr, thr = roc_curve(y_true, scores)
roc_auc = roc_auc_score(y_true, scores)

anomaly_score_figure, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(normal_errors.numpy(),  bins=40, alpha=0.6, density=True, label='normal', color='tab:blue')
ax[0].hist(anomaly_errors.numpy(), bins=40, alpha=0.6, density=True, label='anomalous', color='tab:red')
ax[0].set_xlabel('per-sample reconstruction MSE'); ax[0].set_ylabel('density')
ax[0].set_title('Reconstruction error: normal vs anomalous'); ax[0].legend()
ax[1].plot(fpr, tpr, color='tab:green', label='AUC = %.3f' % roc_auc)
ax[1].plot([0, 1], [0, 1], '--', color='gray', label='chance')
ax[1].set_xlabel('false positive rate'); ax[1].set_ylabel('true positive rate')
ax[1].set_title('ROC: reconstruction error as anomaly score'); ax[1].legend(loc='lower right')
plt.tight_layout(); plt.show()
print('AUC=%.3f | mean normal err=%.4f | mean anomaly err=%.4f'
      % (roc_auc, normal_errors.mean(), anomaly_errors.mean()))

# sanity checks
assert normal_errors.shape[0] == N_NORMAL_TEST and anomaly_errors.shape[0] == N_ANOM_TEST
assert 0.0 <= roc_auc <= 1.0 and math.isfinite(roc_auc)
assert anomaly_errors.mean() > normal_errors.mean() and roc_auc > 0.5

In [ ]:
# Interactive anomaly-threshold explorer
ne = normal_errors.numpy()
ae = anomaly_errors.numpy()
lo = float(min(ne.min(), ae.min()))
hi = float(max(ne.max(), ae.max()))
thr0 = float(np.percentile(ne, 95))

set_seed(SEED)
_norm_idx = np.random.choice(len(ne), 4, replace=False)
_anom_idx = np.random.choice(len(ae), 4, replace=False)
_example_imgs   = [normal_test_images[i] for i in _norm_idx] + [anomalous_test_images[i] for i in _anom_idx]
_example_scores = [float(ne[i]) for i in _norm_idx] + [float(ae[i]) for i in _anom_idx]
_example_kind   = ['normal'] * 4 + ['anomaly'] * 4

def show_threshold(threshold=thr0):
    threshold = float(min(max(threshold, lo), hi))
    TP = int((ae > threshold).sum()); FN = int((ae <= threshold).sum())
    FP = int((ne > threshold).sum()); TN = int((ne <= threshold).sum())
    TPR = TP / (TP + FN) if (TP + FN) else 0.0
    FPR = FP / (FP + TN) if (FP + TN) else 0.0
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    flagged = TP + FP
    fig = plt.figure(figsize=(13, 3.8))
    gs = fig.add_gridspec(2, 7)
    axh = fig.add_subplot(gs[:, 0:3])
    axh.hist(ne, bins=40, alpha=0.6, density=True, label='normal', color='tab:blue')
    axh.hist(ae, bins=40, alpha=0.6, density=True, label='anomalous', color='tab:red')
    axh.axvline(threshold, color='black', linestyle='--', label='thr=%.4f' % threshold)
    axh.set_xlabel('reconstruction MSE'); axh.set_ylabel('density')
    axh.set_title('TPR=%.2f FPR=%.2f precision=%.2f flagged=%d' % (TPR, FPR, precision, flagged))
    axh.legend(fontsize=8)
    for k in range(8):
        a = fig.add_subplot(gs[k // 4, 3 + (k % 4)])
        sc = _example_scores[k]
        flag = 'FLAG' if sc > threshold else 'PASS'
        a.imshow(_example_imgs[k].squeeze(), cmap='gray'); a.axis('off')
        a.set_title('%s %.3f %s' % (_example_kind[k], sc, flag), fontsize=6,
                    color=('tab:red' if flag == 'FLAG' else 'tab:green'))
    plt.show()

show_threshold(thr0)   # default static render

anomaly_threshold_widget = None
if WIDGETS_OK:
    try:
        anomaly_threshold_widget = interact(
            show_threshold,
            threshold=widgets.FloatSlider(min=lo, max=hi, step=(hi - lo) / 100.0, value=thr0),
        )
    except Exception as e:
        print('Widget init failed; static view shown above.', e)

# sanity checks
def _flagged_at(t):
    return int((ae > t).sum() + (ne > t).sum())
assert _flagged_at(lo) >= _flagged_at(hi)
assert lo <= thr0 <= hi

## Wrap-up -- One Mechanism, Four Superpowers

We toured the whole auto-encoder story on **one small backbone**:

| Concept | What we did | Idea |
|---|---|---|
| 1. Reconstruction | trained core_ae; input vs reconstruction | encoder -> bottleneck -> decoder + MSE |
| 2. Latent code | AE code vs PCA vs t-SNE; latent interpolation | the code as learned dimension reduction |
| 3. De-noising / BERT | corrupt then reconstruct-clean; one-line corrupt_fn | de-noising AE = masked self-supervision |
| 4. Anomaly detection | reconstruction error as a score; ROC/AUC; threshold | one-class detection from normal-only data |

**The unifying theme:** de-noising, representation learning, and anomaly detection are all **variants of one idea -- reconstruction through a bottleneck**. A single model trained once (Concepts 1, 2, 4), plus a one-line corruption change (Concept 3), did all of it -- the essence of the **self-supervised** paradigm.

### Where to go next (an advanced follow-up notebook)
- **Variational Auto-encoder (VAE)** -- turn the decoder into a *generator* by learning a probabilistic latent space (KL + reparameterization).
- **Feature disentanglement / voice conversion** -- split the code into content vs. style factors.
- **VQ-VAE** -- discrete latent codes via a learned codebook, a backbone of many modern generative systems.

> Re-run with **Runtime -> Run all**: every result reproduces deterministically (fixed seeds), and each interactive slider re-renders live.